## Exercise 4.2: Design of Recursive Filters

The goal of this exercise is to design a digital lowpass system based on a given tolerance scheme for its amplitude response. There are Python functions, e.g. `butter`, `cheby1`, etc., available in `scipy.signal` to realize a lowpass design for continuous time systems. The tolerance schema, that is given in the Z domain, has to be transformed to the S domain. The known algorithms are used to design the desired filter in the S domain. Then the system function is transformed back to the Z domain.

To transform from Z- to S-domain and vice versa, the **bilinear transformation** is used, which transforms stable systems in the S-domain to stable systems in the Z-domain (Section 4.2.1). Using the bilinear transformation, a predistortion of the $\Omega$-axis in the Z-domain is necessary. The design may be done using the standard procedures described in Section 4.2.2 and 4.2.3 using an auxiliary function.

For the lowpass-design, the functions provided by Python (`scipy.signal`) are represented in the following table. Calculation of the necessary filter order and the normalized cut-off frequency\footnote{All design functions use normalized frequencies between 0 and 1. Normalization is done according to half of the sampling frequency. The normalized cut-off frequency indicates the frequency with $3 \, \text{dB}$ attenuation. Example: Sampling frequency 1000 Hz, cut-off frequency 300 Hz, normalized cut-off frequency: `Wn=300/500=0.6`} can be done using these functions:

| Filter Type        | Determination of Filter Order                                      |
| ------------------ | ------------------------------------------------------------------ |
| Butterworth        | `buttord(Wp, Ws, Rp, Rs, fs=fs)`                      |
| Chebyshev type I   | `cheb1ord(Wp, Ws, Rp, Rs, fs=fs)`                     |
| Chebyshev type II  | `cheb2ord(Wp, Ws, Rp, Rs, fs=fs)`                     |
| Cauer              | `ellipord(Wp, Ws, Rp, Rs, fs=fs)`                     |

Subsequently, the coefficients can be calculated using the design functions:

| Filter Type        | Design Function                                                        |
| ------------------ | ---------------------------------------------------------------------- |
| Butterworth        | `butter(m, Wn, btype='low', output='ba', fs=fs)`          |
| Chebyshev type I   | `cheby1(m, Rp, Wn, btype='low', output='ba', fs=fs)`      |
| Chebyshev type II  | `cheby2(m, Rs, Wn, btype='low', output='ba', fs=fs)`      |
| Cauer              | `ellip(m, Rp, Rs, Wn, btype='low', output='ba', fs=fs)`   |

The meaning of the parameters are:

| Parameter          | Description                                        |
| ------------------ | -------------------------------------------------- |
| `b, a`             | vectors with the filter coefficients               |
| `m`                | order of the filter                                |
| `Wn`               | normalized cut-off frequency of the lowpass        |
| `Wp`               | normalized pass-band frequency of the lowpass      |
| `Ws`               | normalized stop-band frequency of the lowpass      |
| `Rp`               | ripple in pass-band (dB)                           |
| `Rs`               | stop-band attenuation (dB)                         |

Determine the minimal filter order needed for each filter type and design the lowpass-filters for the following tolerance scheme:

- $f_\text{D} = 1 \, \text{kHz}$
- $f_\text{S} = 1.4 \, \text{kHz}$
- $R_\text{P} = 0.5 \, \text{dB}$
- $R_\text{S} = 30 \, \text{dB}$
- $f_A = 8 \, \text{kHz}$

![Tolerance Scheme](toleranz.eps)

Compare the amplitude response and the pole-zero plot of the different filters to each other.

In [ ]:
# BEGIN SOLUTION

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import buttord, butter, cheb1ord, cheby1, cheb2ord, cheby2, ellipord, ellip, freqz
from scipy.signal import tf2zpk as zplane 

fD = 1000     # Passband frequency (Hz)
fS = 1400     # Stopband frequency (Hz)
Rp = 0.5      # Passband ripple (dB)
Rs = 30       # Stopband attenuation (dB)
fA = 8000     # Sampling frequency (Hz)
Wp = 2 * fD / fA  # Conversion to normalized cutoff frequencies
Ws = 2 * fS / fA

for typ in range(1, 5):
    if typ == 1:
        n, Wn = buttord(Wp, Ws, Rp, Rs)  # Butterworth
        b, a = butter(n, Wn)
        FTyp = 'Butterworth'
    elif typ == 2:
        n, Wn = cheb1ord(Wp, Ws, Rp, Rs)  # Chebyshev Type I
        b, a = cheby1(n, Rp, Wn)
        FTyp = 'Chebyshev Type I'
    elif typ == 3:
        n, Wn = cheb2ord(Wp, Ws, Rp, Rs)  # Chebyshev Type II
        b, a = cheby2(n, Rs, Wn)
        FTyp = 'Chebyshev Type II'
    elif typ == 4:
        n, Wn = ellipord(Wp, Ws, Rp, Rs)  # Cauer
        b, a = ellip(n, Rp, Rs, Wn)
        FTyp = 'Cauer'
    else:
        raise ValueError('Something is wrong...')

    # Graphical Output
    TitleString = 'Filter Type: {}  Order: {}'.format(FTyp, n)

    # Frequency Response
    W = np.linspace(0, np.pi, 512)
    _, Hg = freqz(b, a, 512)

    plt.figure()
    plt.plot(W / np.pi, np.abs(Hg))
    plt.grid(True)
    plt.title(TitleString)
    plt.xlabel(r'$\Omega/\pi$')
    plt.ylabel(r'$|H(e^{j\Omega})|$')
    plt.axis([0, 1, 0, 1.2])


    # Draw the tolerance tube
    xline = [Wp, Wp]
    yline = [0, 10 ** (-Rp / 20)]
    plt.plot(xline, yline, 'r--')
    xline = [0, Ws]
    yline = [1, 1]
    plt.plot(xline, yline, 'r--')
    xline = [0, Wp]
    yline = [10 ** (-Rp / 20), 10 ** (-Rp / 20)]
    plt.plot(xline, yline, 'r--')
    xline = [Ws, Ws]
    yline = [10 ** (-Rs / 20), 1]
    plt.plot(xline, yline, 'r--')
    xline = [Ws, 1]
    yline = [10 ** (-Rs / 20), 10 ** (-Rs / 20)]
    plt.plot(xline, yline, 'r--')

    # Pole and zero plot in the z-plane
    plt.figure()
    zplane(b, a)
    plt.title(TitleString)
    plt.xlabel('Real Part')
    plt.ylabel('Imaginary Part')

plt.show()

# END SOLUTION